In [27]:
import httpx, json

def chat(prompt, max_tokens=2000, thinking=True):
    with httpx.Client(timeout=httpx.Timeout(connect=30, read=300, write=30, pool=30)) as c:
        body = {
            "model": "/models/qwen3.6-27b-fp8",
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
            "stream": True,
            "temperature": 0.2,
        }
        if not thinking:
            body["chat_template_kwargs"] = {"enable_thinking": False}

        with c.stream("POST", "http://10.27.40.222:8000/v1/chat/completions",
                      headers={"Content-Type": "application/json"},
                      content=json.dumps(body)) as r:
            in_thinking = False
            for line in r.iter_lines():
                if not line.startswith("data:") or "[DONE]" in line:
                    continue
                delta = json.loads(line[5:])["choices"][0]["delta"]

                if thinking and (t := delta.get("reasoning")):
                    if not in_thinking:
                        print("💭 Thinking:\n")
                        in_thinking = True
                    print(t, end="", flush=True)

                if a := delta.get("content"):
                    if in_thinking:
                        print("\n\n" + "─"*60 + "\n")
                        in_thinking = False
                    print(a, end="", flush=True)
    print()

file = "../knowledge_base/knowledge/cdm_data_path_tree.json"
with open(file, "r") as f:
    data = json.load(f)
print(str(data))
chat(str(data) + "Fait moi un résumé de ce que tu as compris.",
     thinking=True)  

{'trade': {'product': {'taxonomy': {'source': None, 'productQualifier': None, 'value': {'name': {'value': None, 'meta': {'scheme': None}}}, 'primaryAssetClass': {'value': None}}, 'economicTerms': {'payout': {'InterestRatePayout': {'payerReceiver': {'payer': None, 'receiver': None}, 'priceQuantity': {'quantitySchedule': {'address': {'scope': None, 'value': None}}, 'meta': {'globalKey': None}, 'quantityReference': {'externalReference': None}, 'quantityMultiplier': {'fxLinkedNotionalSchedule': {'varyingNotionalCurrency': {'value': None}, 'varyingNotionalFixingDates': {'periodMultiplier': None, 'period': None, 'meta': {'globalKey': None}, 'dayType': None, 'businessDayConvention': None, 'businessCenters': {'businessCenter': {'value': None}, 'meta': {'globalKey': None}}, 'dateRelativeTo': {'globalReference': None, 'externalReference': None}}, 'fxSpotRateSource': {'primarySource': {'sourceProvider': {'value': None}}}, 'fixingTime': {'hourMinuteTime': None, 'businessCenter': {'value': None}}, 

## Utiliser un MCP server avec vLLM (côté client uniquement)

**Principe** : vLLM expose une API OpenAI-compatible qui supporte le *tool calling*.  
On utilise le package `mcp` comme **client** pour :
1. Se connecter au serveur MCP (transport StreamableHTTP ou SSE)
2. Récupérer la liste des outils et les convertir en format OpenAI `tools`
3. Envoyer la requête à vLLM avec les outils
4. Si vLLM répond avec un `tool_call` → l'exécuter sur le MCP → renvoyer le résultat
5. Répéter jusqu'à réponse finale (boucle agentique)


In [3]:
%pip install mcp python-dotenv --quiet


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import json
import os
import httpx
from dotenv import load_dotenv
from openai import OpenAI
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

load_dotenv(dotenv_path=r"d:\OneDrive - Murex\Documents\interface_generation\.env")

# ── Config ────────────────────────────────────────────────────────────────────
MCP_URL   = os.environ["TAVILY_MCP"]          # StreamableHTTP endpoint
LLM_MODEL = "/models/qwen3.6-27b-fp8"
VLLM_URL  = "http://10.27.40.184:8000/v1"

llm = OpenAI(
    base_url=VLLM_URL,
    api_key="dummy",
    http_client=httpx.Client(timeout=httpx.Timeout(connect=30.0, read=1800.0, write=30.0, pool=30.0)),
)

print("Config OK")
print(f"  MCP  → {MCP_URL[:60]}...")
print(f"  vLLM → {VLLM_URL}")


Config OK
  MCP  → https://mcp.tavily.com/mcp/?tavilyApiKey=tvly-dev-3Xv8jM-CvJ...
  vLLM → http://10.27.40.184:8000/v1


In [27]:
# ── Inspecter les outils disponibles sur le MCP server ───────────────────────
async with streamablehttp_client(MCP_URL) as (read, write, _):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools_result = await session.list_tools()

for t in tools_result.tools:
    print(f"🔧 {t.name}")
    print(f"   {t.description}")
    print()


🔧 tavily_search
   Search the web for current information on any topic. Use for news, facts, or data beyond your knowledge cutoff. Returns snippets and source URLs.

🔧 tavily_extract
   Extract content from URLs. Returns raw page content in markdown or text format.

🔧 tavily_crawl
   Crawl a website starting from a URL. Extracts content from pages with configurable depth and breadth.

🔧 tavily_map
   Map a website's structure. Returns a list of URLs found starting from the base URL.

🔧 tavily_research
   Perform comprehensive research on a given topic or question. Use this tool when you need to gather information from multiple sources, including web pages, documents, and other resources, to answer a question or complete a task. Returns a detailed response based on the research findings. Rate limit: 20 requests per minute.



In [ ]:
# ── Boucle agentique : vLLM + MCP tools ──────────────────────────────────────
import re
from datetime import date

def mcp_tools_to_openai(mcp_tools: list) -> list:
    """Convertit les outils MCP en format OpenAI function calling."""
    return [
        {
            "type": "function",
            "function": {
                "name": t.name,
                "description": t.description or "",
                "parameters": t.inputSchema,
            },
        }
        for t in mcp_tools
    ]


def parse_text_tool_calls(content: str) -> list[dict] | None:
    """
    Fallback : certains modèles retournent les tool calls en XML dans le content
    au lieu d'utiliser l'API OpenAI tool_calls.
    """
    blocks = re.findall(r"<tool_call>(.*?)</tool_call>", content, re.DOTALL)
    if not blocks:
        return None

    calls = []
    for block in blocks:
        fn_match = re.search(r"<function=(\w+)>", block)
        if not fn_match:
            continue
        fn_name = fn_match.group(1)
        params = {}
        for pm in re.finditer(r"<parameter=(\w+)>\s*(.*?)\s*</parameter>", block, re.DOTALL):
            key, val = pm.group(1), pm.group(2).strip()
            try:
                params[key] = json.loads(val)
            except (json.JSONDecodeError, ValueError):
                params[key] = val
        calls.append({"name": fn_name, "arguments": params})
    return calls if calls else None


async def mcp_chat(prompt: str, max_iterations: int = 5) -> None:
    """
    Envoie un prompt à vLLM avec les outils du MCP server.
    Gère la boucle tool_call → exécution MCP → réponse finale.
    Supporte les tool calls via l'API OpenAI ET via le format XML texte.
    """
    async with streamablehttp_client(MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools_result = await session.list_tools()
            openai_tools = mcp_tools_to_openai(tools_result.tools)
            print(f"📦 {len(openai_tools)} outil(s) chargé(s) depuis le MCP\n")

            # ── System prompt avec la date courante ──────────────────────────
            today = date.today().strftime("%d %B %Y")
            system_prompt = (
                f"Tu es un assistant utile. "
                f"Aujourd'hui nous sommes le {today}. "
                f"Ta date de coupure d'entraînement est antérieure à cette date : "
                f"utilise les outils disponibles pour obtenir des informations récentes."
            )

            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": prompt},
            ]

            # Boucle agentique
            for iteration in range(max_iterations):
                response = llm.chat.completions.create(
                    model=LLM_MODEL,
                    messages=messages,
                    tools=openai_tools,
                    tool_choice="auto",
                    max_tokens=2000,
                    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
                )
                msg = response.choices[0].message
                content = msg.content or ""

                # Cas 1 : tool calls via l'API OpenAI (format standard)
                if msg.tool_calls:
                    tool_calls_to_run = [
                        {"id": tc.id, "name": tc.function.name,
                         "arguments": json.loads(tc.function.arguments)}
                        for tc in msg.tool_calls
                    ]
                    messages.append(msg.model_dump(exclude_unset=True))

                # Cas 2 : tool calls en XML dans le content (fallback)
                elif text_calls := parse_text_tool_calls(content):
                    print("⚠️  Tool calls détectés en format texte XML (fallback)\n")
                    tool_calls_to_run = [
                        {"id": f"call_{i}", **tc}
                        for i, tc in enumerate(text_calls)
                    ]
                    messages.append({"role": "assistant", "content": content})

                # Cas 3 : réponse finale
                else:
                    print("🤖 Réponse finale :\n")
                    print(content)
                    return

                # Exécuter les tool calls sur le MCP
                for tc in tool_calls_to_run:
                    print(f"🔧 [{iteration+1}] {tc['name']}({json.dumps(tc['arguments'], ensure_ascii=False)[:120]})")
                    result = await session.call_tool(tc["name"], tc["arguments"])
                    result_text = result.content[0].text if result.content else "(aucun résultat)"
                    print(f"   ↳ {result_text[:200]}...\n")

                    messages.append({
                        "role": "tool",
                        "tool_call_id": tc["id"],
                        "content": result_text,
                    })

            print("⚠️  Limite d'itérations atteinte.")


# ── Lancer une requête ────────────────────────────────────────────────────────
await mcp_chat("Quelles sont les dernières actualités sur le CDM FINOS en 2026 ?")


📦 5 outil(s) chargé(s) depuis le MCP

⚠️  Tool calls détectés en format texte XML (fallback)

🔧 [1] tavily_search({"query": "CDM FINOS 2026 latest news updates", "max_results": 10, "search_depth": "advanced", "time_range": "year"})
   ↳ {"query":"CDM FINOS 2026 latest news updates","follow_up_questions":null,"answer":null,"images":[],"results":[{"url":"https://www.finos.org/blog/from-standards-to-impact-cdm-becomes-an-active-finos-pr...

🔧 [1] tavily_search({"query": "FINOS Common Domain Model CDM 2026", "max_results": 10, "search_depth": "advanced", "time_range": "year"})
   ↳ {"query":"FINOS Common Domain Model CDM 2026","follow_up_questions":null,"answer":null,"images":[],"results":[{"url":"https://www.finos.org/this-week-at-finos/week-of-may-11-2026","title":"This Week A...



## Appel via LM Studio (local)

LM Studio expose une API OpenAI-compatible sur `http://localhost:1234/v1`.


In [9]:
import httpx
from openai import OpenAI

LMS_URL   = "http://10.27.40.184:12345/v1"
LMS_MODEL = "qwen3.6-40b-claude-4.6-opus-deckard-heretic-uncensored-thinking-neo-code-di-imatrix-max"

lms = OpenAI(
    base_url=LMS_URL,
    api_key="lm-studio",
    http_client=httpx.Client(timeout=httpx.Timeout(connect=30.0, read=1800.0, write=30.0, pool=30.0))
)

print(f"✅ Connecté à {LMS_URL}")
print(f"✅ Modèle : {LMS_MODEL}")


✅ Connecté à http://10.27.40.184:12345/v1
✅ Modèle : qwen3.6-40b-claude-4.6-opus-deckard-heretic-uncensored-thinking-neo-code-di-imatrix-max


In [9]:
stream = lms.chat.completions.create(
    model=LMS_MODEL,
    messages=[{"role": "user", "content": "Bonjour, présente-toi en 2 phrases."}],
    max_tokens=500,
    stream=True,
)

for chunk in stream:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)
print()




Bonjour ! Je suis Qwen, un grand modèle de langage développé par le laboratoire Tongyi d'Alibaba Group. Mon rôle est de vous aider à répondre à vos questions et à résoudre divers problèmes grâce au traitement du langage naturel.


In [8]:

# ── Gemma-4-e4b via LM Studio ─────────────────────────────────────────────────
GEMMA_MODEL = "google/gemma-4-e4b"

stream = lms.chat.completions.create(
    model=GEMMA_MODEL,
    messages=[{"role": "user", "content": "Bonjour, présente-toi en 2 phrases."}],
    max_tokens=500,
    stream=True,
)

for chunk in stream:
    content = chunk.choices[0].delta.content
    if content:
        print(content, end="", flush=True)
print()


NameError: name 'lms' is not defined

In [1]:
import json
import httpx

VLLM_BASE_URL = "http://10.27.40.222:12345/v1"
VLLM_MODEL = "google/gemma-4-26b-a4b-qat:2"

payload = {
    "model": VLLM_MODEL,
    "messages": [
        {
            "role": "user",
            "content": "Hello! What model are you and what context length do you support?",
        }
    ],
    "temperature": 0.0,
    "max_tokens": 100,
}

resp = httpx.post(
    f"{VLLM_BASE_URL}/chat/completions",
    headers={"Content-Type": "application/json"},
    json=payload,
    timeout=httpx.Timeout(connect=30.0, read=300.0, write=30.0, pool=30.0),
)

resp.raise_for_status()
print(json.dumps(resp.json(), indent=2, ensure_ascii=False))

{
  "id": "chatcmpl-xa03ab4slf7b0w7t3mayr",
  "object": "chat.completion",
  "created": 1781513681,
  "model": "google/gemma-4-26b-a4b-qat:2",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "",
        "reasoning_content": "\n*   Question 1: \"What model are you?\"\n    *   Question 2: \"What context length do you support?\"\n\n    *   I am a large language model, trained by Google.\n    *   *Self-Correction/Refinement:* I don't have a specific \"name\" like \"GPT-4\" or \"Claude 3\" that I am instructed to use unless I am a specific version of Gemini. However, the standard",
        "tool_calls": []
      },
      "logprobs": null,
      "finish_reason": "length"
    }
  ],
  "usage": {
    "prompt_tokens": 30,
    "completion_tokens": 100,
    "total_tokens": 130,
    "completion_tokens_details": {
      "reasoning_tokens": 97
    }
  },
  "stats": {},
  "system_fingerprint": "google/gemma-4-26b-a4b-qat:2"
}


In [3]:
import json
import httpx

VLLM_BASE_URL = "http://10.27.40.222:12345/v1"
VLLM_MODEL = "google/gemma-4-26b-a4b-qat:2"

stream_payload = {
    "model": VLLM_MODEL,
    "messages": [
        {
            "role": "user",
            "content": "Hello! What model are you and what context length do you support?",
        }
    ],
    "temperature": 0.0,
    "max_tokens": 30000,
    "stream": True,
}

with httpx.Client(timeout=httpx.Timeout(connect=30.0, read=300.0, write=30.0, pool=30.0)) as client:
    with client.stream(
        "POST",
        f"{VLLM_BASE_URL}/chat/completions",
        headers={"Content-Type": "application/json"},
        json=stream_payload,
    ) as resp:
        resp.raise_for_status()

        printed_reasoning_header = False
        printed_answer_header = False

        for line in resp.iter_lines():
            if not line or not line.startswith("data:"):
                continue

            data = line[len("data:"):].strip()
            if data == "[DONE]":
                break

            chunk = json.loads(data)
            delta = chunk["choices"][0].get("delta", {})

            reasoning = delta.get("reasoning_content") or delta.get("reasoning")
            if reasoning:
                if not printed_reasoning_header:
                    print("\n--- reasoning ---\n")
                    printed_reasoning_header = True
                print(reasoning, end="", flush=True)

            content = delta.get("content")
            if content:
                if not printed_answer_header:
                    print("\n\n--- answer ---\n")
                    printed_answer_header = True
                print(content, end="", flush=True)

print("\n\n[DONE]")


--- reasoning ---


*   Question 1: "What model are you?"
    *   Question 2: "What context length do you support?"

    *   I am a large language model, trained by Google.
    *   *Self-Correction/Refinement:* I don't have a specific "name" like "GPT-4" or "Claude 3" that I am instructed to use unless I am a specific version of Gemini. However, the standard response for a Google-trained model is to identify as a large language model trained by Google.

    *   Context length varies depending on the specific version/deployment of the model (e.g., Gemini Pro, Gemini Ultra, Gemini Flash).
    *   *Standard Knowledge:* Gemini models are known for having very large context windows (up to 1M or even 2M tokens in some versions).
    *   *Constraint:* As an AI, I don't always know the *exact* technical limit of the specific instance the user is interacting with at this very second, but I can provide general information about the Gemini family.

    *   *Draft 1 (Internal Monologue):* I am a 


## MCP multi-sources : stdio + HTTP dans une seule boucle agentique

Approche générique qui gère **n'importe quelle combinaison** de serveurs MCP (locaux via stdio, distants via HTTP) sans dupliquer la boucle agentique.

- Les sources sont déclarées dans une liste de dicts
- `AsyncExitStack` ouvre toutes les connexions en même temps
- Les outils sont agrégés et un `dict` mappe chaque nom d'outil → sa session MCP


In [28]:

import json
import re
import os
import httpx
from contextlib import AsyncExitStack
from dotenv import load_dotenv
from openai import AsyncOpenAI
from mcp import ClientSession
from mcp.client.sse import sse_client
from mcp.client.streamable_http import streamablehttp_client

load_dotenv(dotenv_path=r"d:\OneDrive - Murex\Documents\interface_generation\.env")

LLM_MODEL = "/models/qwen3.6-27b-fp8"
VLLM_URL  = "http://10.27.40.222:8000/v1"

# AsyncOpenAI : ne bloque pas l'event loop → les connexions SSE restent vivantes
llm = AsyncOpenAI(
    base_url=VLLM_URL,
    api_key="dummy",
    http_client=httpx.AsyncClient(timeout=httpx.Timeout(connect=30.0, read=1800.0, write=30.0, pool=30.0)),
)

# ── Déclaration des sources MCP ───────────────────────────────────────────────
#
# stdio_client est cassé sous Python 3.14 + Jupyter/Windows.
# Toutes les sources passent par HTTP.
#
# Pour le serveur filesystem local, lancer supergateway en mode StreamableHTTP
# (évite le crash "Already connected" du mode SSE lors des reconnexions) :
#
#   npx -y supergateway --port 8080 --outputTransport streamableHttp ^
#     --stdio "npx -y @modelcontextprotocol/server-filesystem .\workspaces"
#
# Endpoint exposé → http://localhost:8080/mcp
#
MCP_SOURCES = [
    {
        "name": "filesystem",
        "type": "streamable_http",
        "url": "http://localhost:8080/mcp",   # supergateway en mode streamableHttp
    },
    {
        "name": "tavily",
        "type": "streamable_http",
        "url": os.environ["TAVILY_MCP"],
    },
]


# ── Helpers ───────────────────────────────────────────────────────────────────
def mcp_tools_to_openai(tools: list) -> list:
    return [
        {
            "type": "function",
            "function": {
                "name": t.name,
                "description": t.description or "",
                "parameters": t.inputSchema,
            },
        }
        for t in tools
    ]


def parse_text_tool_calls(content: str) -> list[dict] | None:
    """
    Fallback : Qwen3 génère les tool calls en XML dans le content quand vLLM
    ne réussit pas à les parser (hermes_tool_parser attend du JSON).
    """
    blocks = re.findall(r"<tool_call>(.*?)</tool_call>", content, re.DOTALL)
    if not blocks:
        return None

    calls = []
    for block in blocks:
        fn_match = re.search(r"<function=(\w+)>", block)
        if not fn_match:
            continue
        fn_name = fn_match.group(1)
        params = {}
        for pm in re.finditer(r"<parameter=(\w+)>\s*(.*?)\s*</parameter>", block, re.DOTALL):
            key, val = pm.group(1), pm.group(2).strip()
            try:
                params[key] = json.loads(val)
            except (json.JSONDecodeError, ValueError):
                params[key] = val
        calls.append({"name": fn_name, "arguments": params})
    return calls if calls else None


async def _open_session(src: dict, stack: AsyncExitStack) -> ClientSession:
    """Ouvre une connexion MCP (SSE ou StreamableHTTP) et retourne la session initialisée."""
    if src["type"] == "sse":
        read, write = await stack.enter_async_context(sse_client(src["url"]))
    elif src["type"] == "streamable_http":
        read, write, _ = await stack.enter_async_context(streamablehttp_client(src["url"]))
    else:
        raise ValueError(f"Type MCP inconnu : {src['type']}")

    session = await stack.enter_async_context(ClientSession(read, write))
    await session.initialize()
    return session


# ── Boucle agentique multi-sources ────────────────────────────────────────────
async def mcp_chat_multi(
    prompt: str,
    sources: list[dict] = MCP_SOURCES,
    max_iterations: int = 20,
) -> None:
    async with AsyncExitStack() as stack:
        tool_to_session: dict[str, ClientSession] = {}
        all_tools = []

        for src in sources:
            session = await _open_session(src, stack)
            tools_result = await session.list_tools()
            for t in tools_result.tools:
                tool_to_session[t.name] = session
                all_tools.append(t)
            print(f"✅ [{src['name']}] {len(tools_result.tools)} outil(s)")

        openai_tools = mcp_tools_to_openai(all_tools)
        print(f"\n📦 Total : {len(openai_tools)} outil(s)\n")

        messages = [{"role": "user", "content": prompt }]

        for iteration in range(max_iterations):
            response = await llm.chat.completions.create(
                model=LLM_MODEL,
                messages=messages,
                tools=openai_tools,
                tool_choice="auto",
                max_tokens=2000,
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},
            )
            msg     = response.choices[0].message
            content = msg.content or ""

            # # Cas 1 : tool calls via l'API OpenAI (format standard)
            # if msg.tool_calls:
            #     messages.append(msg.model_dump(exclude_unset=True))
            #     for tc in msg.tool_calls:
            #         name      = tc.function.name
            #         arguments = json.loads(tc.function.arguments)
            #         session   = tool_to_session[name]
            #         print(f"🔧 [{iteration+1}] {name}({json.dumps(arguments, ensure_ascii=False)[:120]})")
            #         result      = await session.call_tool(name, arguments)
            #         result_text = result.content[0].text if result.content else "(aucun résultat)"
            #         print(f"   ↳ {result_text[:200]}...\n")
            #         messages.append({
            #             "role": "tool",
            #             "tool_call_id": tc.id,
            #             "content": result_text,
            #         })
            print(f"{iteration+1}. {content[:100]}...\n")

            # Cas 2 : tool calls en XML dans le content (fallback Qwen3 / hermes mismatch)
            if text_calls := parse_text_tool_calls(content):
                print("⚠️  Tool calls XML détectés (fallback)\n")
                messages.append({"role": "assistant", "content": content})
                for i, tc in enumerate(text_calls):
                    name      = tc["name"]
                    arguments = tc["arguments"]
                    session   = tool_to_session.get(name)
                    if session is None:
                        print(f"⚠️  Outil inconnu : {name}")
                        continue
                    print(f"🔧 [{iteration+1}] {name}({json.dumps(arguments, ensure_ascii=False)[:120]})")
                    result      = await session.call_tool(name, arguments)
                    result_text = result.content[0].text if result.content else "(aucun résultat)"
                    print(f"   ↳ {result_text[:200]}...\n")
                    messages.append({
                        "role": "tool",
                        "tool_call_id": f"call_{iteration}_{i}",
                        "content": result_text,
                    })

            # Cas 3 : réponse finale
            else:
                print("🤖 Réponse finale :\n")
                print(content)
                return

        print("⚠️  Limite d'itérations atteinte.")



In [29]:

# # Recherche web + lecture de fichiers locaux en une seule requête
# await mcp_chat_multi(
#     "Cherche les dernières actualités sur Roland Garros en 2026, "
#     r"puis sauvegarde un résumé dans .\workspaces\cdm_news.txt"
# )
await mcp_chat_multi(
    "est ce que tu peux lire les fichiers que j'ai dans workspaces et me faire un résumé ?"
)


  + Exception Group Traceback (most recent call last):
  |   File "C:\Users\cleberre\AppData\Roaming\Python\Python314\site-packages\IPython\core\interactiveshell.py", line 3745, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "C:\Users\cleberre\AppData\Local\Temp\ipykernel_31068\3762994766.py", line 6, in <module>
  |     await mcp_chat_multi(
  |         "est ce que tu peux lire les fichiers que j'ai dans workspaces et me faire un résumé ?"
  |     )
  |   File "C:\Users\cleberre\AppData\Local\Temp\ipykernel_31068\2921149870.py", line 112, in mcp_chat_multi
  |     async with AsyncExitStack() as stack:
  |                ~~~~~~~~~~~~~~^^
  |   File "c:\Users\cleberre\AppData\Local\Python\pythoncore-3.14-64\Lib\contextlib.py", line 768, in __aexit__
  |     raise exc
  |   File "c:\Users\cleberre\AppData\Local\Python\pythoncore-3.14-64\Lib\contextlib.py", line 751, in __aexit__
  |     cb_suppress = await cb(*exc_details)
  |                   ^^^